# 55 Input & Result — Estimador de Precio

Replica los modelos de **`55_sale_rent_models`** leyendo la configuración directamente de `data/model_results/params_sale.json` y `params_rent.json`.

| Modelo | Fuente | Test R² |
|---|---|---|
| XGBoost Sale (`53_boost_sale_optuna`) | `params_sale.json` | ≈ 0.85 |
| XGBoost Rent (`53_boost_rent`) | `params_rent.json` | ≈ 0.62 |

Ejecuta todas las celdas en orden y modifica la **última celda** para introducir los datos.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "gold").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/gold)")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PARAMS_DIR   = PROJECT_ROOT / "data" / "model_results"
print(f"Proyecto: {PROJECT_ROOT}")


Proyecto: /Users/sitomachucas/Documents/BezanillaSL


In [2]:
sale_cfg = json.loads((PARAMS_DIR / "params_sale.json").read_text(encoding="utf-8"))
rent_cfg = json.loads((PARAMS_DIR / "params_rent.json").read_text(encoding="utf-8"))

RANDOM_STATE = sale_cfg["random_state"]
TEST_SIZE    = sale_cfg["test_size"]

SALE_TARGET_COL     = sale_cfg["target_col"]
SALE_MIN_MUNI_OBS   = sale_cfg["min_muni_obs"]
SALE_BASE_FEATURES  = sale_cfg["base_features"]
XGB_PARAMS_SALE     = {**sale_cfg["xgb_params"],
                       "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": 0}

RENT_TARGET_COL           = rent_cfg["target_col"]
RENT_MIN_MUNI_OBS         = rent_cfg["min_muni_obs"]
RENT_BASE_FEATURES        = rent_cfg["base_features"]
XGB_PARAMS_RENT           = {**rent_cfg["xgb_params"],
                              "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": 0}

SALE_PATH = PROJECT_ROOT / "data" / "gold" / "final_sale_idealistaAPI.csv"
RENT_PATH = PROJECT_ROOT / "data" / "gold" / "final_rent_idealistaAPI.csv"
PROC_RENT_PATH = PROJECT_ROOT / "data" / "processed" / "idealistaAPI" / "total_rent_cantabria_outliers.csv"

s_nb  = sale_cfg["notebook"];  r_nb  = rent_cfg["notebook"]
s_tri = sale_cfg["optuna_best_trial"]; r_tri = rent_cfg["optuna_best_trial"]
s_cv  = sale_cfg["optuna_cv_rmse"];   r_cv  = rent_cfg["optuna_cv_rmse"]
s_r2  = sale_cfg["test_r2"];          r_r2  = rent_cfg["test_r2"]
print(f"Sale [{s_nb}] trial #{s_tri}  CV-RMSE={s_cv}  Test R\u00b2={s_r2}")
print(f"Rent [{r_nb}] trial #{r_tri}  CV-RMSE={r_cv}  Test R\u00b2={r_r2}")


Sale [53_boost_sale_optuna] trial #68  CV-RMSE=0.23445  Test R²=0.8313
Rent [53_boost_rent] trial #62  CV-RMSE=0.14791  Test R²=0.59922


In [3]:
def build_X(df: pd.DataFrame, base_features: list, min_muni_obs: int) -> tuple:
    df2 = df.copy()
    base = [f for f in base_features if f in df2.columns]
    mun_cols = sorted([c for c in df2.columns if c.startswith("municipio_")])
    if mun_cols:
        counts = df2[mun_cols].sum()
        small  = counts[counts < min_muni_obs].index.tolist()
        if small:
            df2["municipio_otros"] = df2[small].max(axis=1)
            df2 = df2.drop(columns=small)
        mun_final = sorted(c for c in df2.columns if c.startswith("municipio_"))
    else:
        mun_final = []
    all_feats = base + [m for m in mun_final if m not in base]
    X_raw = df2[all_feats].copy()

    # Piso-only features → NaN para unifamiliares; XGBoost aprende a enrutarlas.
    _PISO_ONLY = {"tiene_ascensor_piso", "es_exterior_piso",
                  "planta_num", "interaccion_planta_sin_ascensor_piso"}
    if "tipologia_unificada_unifamiliar" in df2.columns:
        _uni = df2["tipologia_unificada_unifamiliar"] == 1
        for _f in _PISO_ONLY:
            if _f in X_raw.columns:
                X_raw.loc[_uni, _f] = np.nan

    medians = X_raw.median().to_dict()  # NaN-aware: medians de piso = solo pisos
    _piso_cols  = [f for f in all_feats if f in _PISO_ONLY]
    _other_cols = [f for f in all_feats if f not in _PISO_ONLY]
    _imp = SimpleImputer(strategy="median")
    _X_other = pd.DataFrame(_imp.fit_transform(X_raw[_other_cols]),
                             columns=_other_cols, index=X_raw.index)
    X = pd.concat([_X_other, X_raw[_piso_cols]], axis=1)[all_feats]
    return X, all_feats, medians

print("Funciones cargadas.")


Funciones cargadas.


In [4]:
# ── SALE ─────────────────────────────────────────────────────────────────────
df_sale = pd.read_csv(SALE_PATH)
df_sale = df_sale[df_sale[SALE_TARGET_COL].notna()].copy()
print(f"Sale  \u2014 filas cargadas: {len(df_sale)} (outliers eliminados upstream)")

X_sale, feats_sale, medians_sale = build_X(df_sale, SALE_BASE_FEATURES, SALE_MIN_MUNI_OBS)
y_sale = df_sale[SALE_TARGET_COL].values
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_sale, y_sale, test_size=TEST_SIZE, random_state=RANDOM_STATE)
model_sale = XGBRegressor(**XGB_PARAMS_SALE)
model_sale.fit(Xs_tr, ys_tr)
sale_rmse_test = float(np.sqrt(mean_squared_error(ys_te, model_sale.predict(Xs_te))))
print(f"Sale  \u2014 entrenado ({len(feats_sale)} features)  |  test RMSE: {sale_rmse_test:.4f}  \u2192  \u00b1{(np.exp(sale_rmse_test)-1)*100:.1f}%")

# ── RENT ─────────────────────────────────────────────────────────────────────
df_rent = pd.read_csv(RENT_PATH)
df_rent = df_rent[df_rent[RENT_TARGET_COL].notna() & df_rent["precio_m2"].notna()].copy()
print(f"Rent  \u2014 filas cargadas: {len(df_rent)} (outliers eliminados upstream)")

X_rent, feats_rent, medians_rent = build_X(df_rent, RENT_BASE_FEATURES, RENT_MIN_MUNI_OBS)
y_rent = df_rent[RENT_TARGET_COL].values
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_rent, y_rent, test_size=TEST_SIZE, random_state=RANDOM_STATE)
model_rent = XGBRegressor(**XGB_PARAMS_RENT)
model_rent.fit(Xr_tr, yr_tr)
rent_rmse_test = float(np.sqrt(mean_squared_error(yr_te, model_rent.predict(Xr_te))))
print(f"Rent  \u2014 entrenado ({len(feats_rent)} features)  |  test RMSE: {rent_rmse_test:.4f}  \u2192  \u00b1{(np.exp(rent_rmse_test)-1)*100:.1f}%")
print("\n\u2713 Ambos modelos listos.")


Sale  — filas cargadas: 2532 (outliers eliminados upstream)
Sale  — entrenado (47 features)  |  test RMSE: 0.2339  →  ±26.4%
Rent  — filas cargadas: 661 (outliers eliminados upstream)
Rent  — entrenado (23 features)  |  test RMSE: 0.1549  →  ±16.8%

✓ Ambos modelos listos.


In [5]:
GEO_COLS = [
    "distancia_min_playa_km", "distancia_min_supermercado_km",
    "distancia_min_colegio_km", "precio_m2_municipio_media",
    "distancia_centro_municipio_km", "score_cercania_servicios",
    "latitud", "longitud",
]

def build_geo_ref(df: pd.DataFrame) -> pd.DataFrame:
    mun_cols = [c for c in df.columns if c.startswith("municipio_")]
    rows = []
    for mc in mun_cols:
        nombre = mc.replace("municipio_", "")
        subset = df[df[mc] == 1]
        if len(subset) == 0:
            continue
        row = {"municipio": nombre}
        for gc in GEO_COLS:
            row[gc] = subset[gc].median() if gc in subset.columns else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("municipio")

sale_geo_ref = build_geo_ref(df_sale)
rent_geo_ref = build_geo_ref(df_rent)

# Sobrescribir precio_m2_municipio_media con las medias del JSON.
# El modelo se entrenó con esas medias: sale→train-only, rent→all-sale.
_s_means  = sale_cfg.get("mun_means_sale", {})
_s_global = sale_cfg.get("mun_global_mean_sale",
                         float(sale_geo_ref["precio_m2_municipio_media"].mean()))
sale_geo_ref["precio_m2_municipio_media"] = sale_geo_ref.index.map(
    lambda m: _s_means.get(m, _s_global)
)

_r_means  = rent_cfg.get("mun_means_sale", {})
_r_global = rent_cfg.get("mun_global_mean_sale",
                         float(rent_geo_ref["precio_m2_municipio_media"].mean()))
rent_geo_ref["precio_m2_municipio_media"] = rent_geo_ref.index.map(
    lambda m: _r_means.get(m, _r_global)
)

# ── Extender rent_geo_ref con municipios agrupados en "otro" ────────────
_proc = pd.read_csv(PROC_RENT_PATH)
_proc["_lat"] = _proc["latitude"].round(5)
_proc["_lon"] = _proc["longitude"].round(5)
_gdf  = pd.read_csv(RENT_PATH)
_gdf["_lat"] = _gdf["latitud"].round(5)
_gdf["_lon"] = _gdf["longitud"].round(5)
_geo_feats = [c for c in GEO_COLS if c != "precio_m2_municipio_media"]
_merged = _proc.merge(_gdf[["_lat","_lon","municipio_otro"] + _geo_feats],
                      on=["_lat","_lon"], how="inner")
_otro_m = _merged[_merged["municipio_otro"] == 1]
_added = 0
for _mun, _grp in _otro_m.groupby("municipality"):
    if _mun in rent_geo_ref.index:
        continue
    _row = {gc: _grp[gc].median() if gc in _grp.columns else np.nan for gc in _geo_feats}
    _row["precio_m2_municipio_media"] = _r_means.get(_mun, _r_global)
    rent_geo_ref.loc[_mun] = _row
    _added += 1
del _proc, _gdf, _otro_m, _merged

print(f"Referencia geográfica \u2014 Sale: {len(sale_geo_ref)} municipios")
print(f"Referencia geográfica \u2014 Rent: {len(rent_geo_ref)} municipios ({_added} municipios de 'otro' a\u00f1adidos)")


Referencia geográfica — Sale: 30 municipios
Referencia geográfica — Rent: 54 municipios (47 municipios de 'otro' añadidos)


In [6]:
print("Municipios disponibles (VENTA):")
print(sorted(sale_geo_ref.index.tolist()))
print()
print("Municipios disponibles (ALQUILER):")
print(sorted(rent_geo_ref.index.tolist()))


Municipios disponibles (VENTA):
['Ampuero', 'Barcena de Cicero', 'Camargo', 'Castro-Urdiales', 'Colindres', 'Cudon', 'El Astillero', 'Guarnizo', 'Laredo', 'Liendo', 'Limpias', 'Marina de Cudeyo', 'Miengo', 'Mogro', 'Noja', 'Ortuella', 'Piélagos', 'Polanco', 'Ribamontan al Mar', 'Ribamontan al Monte', 'Santa Cruz de Bezana', 'Santander', 'Santoña', 'Santurtzi', 'Suances', 'Torrelavega', 'Villaescusa', 'Viveda', 'Voto', 'otro']

Municipios disponibles (ALQUILER):
['Alfoz de Lloredo', 'Ampuero', 'Barcena de Cicero', 'Beranga', 'Camargo', 'Campoo de Enmedio', 'Cartes', 'Castañeda', 'Castro-Urdiales', 'Colindres', 'Comillas', 'Cudon', 'El Astillero', 'Entrambasaguas', 'Gallarta', 'Guarnizo', 'Guriezo', 'Hazas de Cesto', 'Laredo', 'Liendo', 'Limpias', 'Los Corrales de Buelna', 'Marina de Cudeyo', 'Miengo', 'Mogro', 'Molledo', 'Noja', 'Piélagos', 'Polanco', 'Puente San Miguel', 'Puente Viesgo', 'Ramales de la Victoria', 'Reinosa', 'Reocin', 'Ribadedeva', 'Ribamontan al Mar', 'Ribamontan al Mo

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
#  ESTIMADOR DE PRECIO — modifica los valores y ejecuta esta celda
# ══════════════════════════════════════════════════════════════════════════════
MUNICIPIO      = "Santander"   # Nombre exacto (ver celda anterior)
SUPERFICIE_M2  = 120            # m² construidos
N_DORMITORIOS  = 3             # Número de dormitorios
N_BANOS        = 2             # Número de baños
TIENE_GARAJE   = True          # True / False
OBRA_NUEVA     = False         # True / False
TIPOLOGIA      = "piso"        # "piso"  o  "unifamiliar"

# ── Solo para PISO (pon None si es unifamiliar) ───────────────────────────────
PLANTA         = 4             # 0=bajo, 1=primera…  → None si unifamiliar
ES_EXTERIOR    = True          # True / False         → None si unifamiliar
TIENE_ASCENSOR = True             # True / False         → None si unifamiliar


def _build_row(municipio, sup, dorm, banos, planta, exterior, ascensor,
               garaje, obra_nueva, tipologia, feature_cols, geo_ref, medians):
    row = pd.Series(np.nan, index=feature_cols)

    def _set(k, v):
        if k in row.index and v is not None:
            row[k] = float(v)

    _set("superficie_construida_m2",       sup)
    _set("numero_dormitorios",             dorm)
    _set("numero_banos",                   banos)
    _set("tiene_garaje",                   int(garaje))
    _set("obra_nueva",                     int(obra_nueva))
    _set("tipologia_unificada_piso",       1 if tipologia == "piso"        else 0)
    _set("tipologia_unificada_unifamiliar", 1 if tipologia == "unifamiliar" else 0)
    _set("planta_num",                     planta)
    _set("es_exterior_piso",    int(exterior) if exterior is not None else None)
    _set("tiene_ascensor_piso", int(ascensor) if ascensor is not None else None)

    if tipologia == "piso" and planta is not None and ascensor is not None:
        _set("interaccion_planta_sin_ascensor_piso", planta * (1 - int(ascensor)))

    # ratio_dormitorios_superficie y ratio_banos_superficie se dejan como NaN
    # → relleno con mediana de entrenamiento abajo.
    # Calcularlos desde el input (dorm/sup) invierte la relación cuando sube
    # la superficie: ratio baja, el modelo predice MENOS alquiler → no monótono.

    if municipio in geo_ref.index:
        for col, val in geo_ref.loc[municipio].to_dict().items():
            _set(col, val)

    for c in [c for c in feature_cols if c.startswith("municipio_")]:
        row[c] = 0.0
    mun_col = "municipio_" + municipio
    if mun_col in row.index:
        row[mun_col] = 1.0
    elif "municipio_otro" in row.index:
        row["municipio_otro"] = 1.0
    elif "municipio_otros" in row.index:
        row["municipio_otros"] = 1.0

    _PISO_ONLY_INF = {"tiene_ascensor_piso", "es_exterior_piso",
                      "planta_num", "interaccion_planta_sin_ascensor_piso"}
    for col in feature_cols:
        if pd.isna(row[col]):
            if tipologia == "unifamiliar" and col in _PISO_ONLY_INF:
                pass  # mantener NaN — XGBoost lo enruta
            else:
                row[col] = medians.get(col, 0)
    return pd.DataFrame([row])


def _rango(precio, rmse_log):
    return precio * np.exp(-rmse_log), precio * np.exp(rmse_log)


def estimar_precio(municipio, sup, dorm, banos, planta, exterior,
                   ascensor, garaje, obra_nueva, tipologia):
    desc = []
    if tipologia == "piso":
        if planta   is not None: desc.append(f"Planta {planta}")
        if exterior is not None: desc.append("exterior" if exterior else "interior")
        if ascensor is not None: desc.append("con ascensor" if ascensor else "sin ascensor")
    if garaje:     desc.append("garaje")
    if obra_nueva: desc.append("obra nueva")

    print()
    print("\u2550" * 58)
    print(f"  {sup} m\u00b2  \u00b7  {dorm} dorm.  \u00b7  {banos} ba\u00f1os  \u2014  {municipio}")
    print(f"  {tipologia.upper()}  \u00b7  {' \u00b7 '.join(desc) if desc else '\u2014'}")
    print("\u2550" * 58)

    if municipio not in sale_geo_ref.index:
        print(f"\n  \u26a0  '{municipio}' no disponible en venta")
    else:
        X_pred = _build_row(municipio, sup, dorm, banos, planta, exterior,
                            ascensor, garaje, obra_nueva, tipologia,
                            feats_sale, sale_geo_ref, medians_sale)
        pv = np.exp(model_sale.predict(X_pred)[0])
        lo, hi = _rango(pv, sale_rmse_test)
        pct = (np.exp(sale_rmse_test) - 1) * 100
        print(f"\n  Precio de venta estimado   : {pv:>12,.0f} \u20ac")
        print(f"  Intervalo error (\u00b11\u03c3)      : [{lo:>10,.0f} \u20ac  \u2014  {hi:>10,.0f} \u20ac]  (\u00b1{pct:.0f}%)")

    if municipio not in rent_geo_ref.index:
        print(f"  \u26a0  '{municipio}' no disponible en alquiler")
    else:
        X_pred = _build_row(municipio, sup, dorm, banos, planta, exterior,
                            ascensor, garaje, obra_nueva, tipologia,
                            feats_rent, rent_geo_ref, medians_rent)
        pr = np.exp(model_rent.predict(X_pred)[0])
        lo, hi = _rango(pr, rent_rmse_test)
        pct = (np.exp(rent_rmse_test) - 1) * 100
        print(f"\n  Alquiler mensual estimado  : {pr:>12,.0f} \u20ac/mes")
        print(f"  Intervalo error (\u00b11\u03c3)      : [{lo:>10,.0f} \u20ac/mes  \u2014  {hi:>10,.0f} \u20ac/mes]  (\u00b1{pct:.0f}%)")

    print()


estimar_precio(
    MUNICIPIO, SUPERFICIE_M2, N_DORMITORIOS, N_BANOS,
    PLANTA, ES_EXTERIOR, TIENE_ASCENSOR, TIENE_GARAJE, OBRA_NUEVA, TIPOLOGIA
)



══════════════════════════════════════════════════════════
  120 m²  ·  3 dorm.  ·  2 baños  —  Santander
  PISO  ·  Planta 4 · exterior · con ascensor · garaje
══════════════════════════════════════════════════════════

  Precio de venta estimado   :      526,056 €
  Intervalo error (±1σ)      : [   416,346 €  —     664,677 €]  (±26%)

  Alquiler mensual estimado  :        1,392 €/mes
  Intervalo error (±1σ)      : [     1,193 €/mes  —       1,626 €/mes]  (±17%)

